In [ ]:
#import data
import pandas as pd
df=pd.read_csv("electricity_bill_dataset.csv")

In [ ]:
df.info()


In [ ]:
df.shape

In [ ]:
#null value treatment
df.isna().sum()

In [ ]:
#feature selection
x=df[['Fan', 'Refrigerator', 'AirConditioner', 'Television', 'Monitor',
       'MotorPump', 'Month', 'City', 'Company', 'MonthlyHours', 'TariffRate']]
y=df['ElectricityBill']

In [ ]:
x

In [ ]:
#preprocessing
cat=[]
num=[]
for i in x.columns:
  if x[i].dtype in ['int64', 'float64']:
    num.append(i)
  else:
    cat.append(i)

Xcat=x[cat]
Xnum=x[num]

from sklearn.preprocessing import LabelEncoder,StandardScaler
le=LabelEncoder()
ss=StandardScaler()

for i in Xcat.columns:
    Xcat[i]=le.fit_transform(Xcat[i])

Xnum=pd.DataFrame(ss.fit_transform(Xnum),columns=num)

x=Xcat.join(Xnum)

In [ ]:
x

In [ ]:

#PICKLE
import pickle
with open('scaler.pkl','wb') as file1:  
    pickle.dump(ss,file1)


In [ ]:
#train-test split 
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=41)

In [ ]:
xtrain.shape

In [ ]:
xtest.shape

#TRAINING MULTIPLE MODELS TO SELECT BEST ONE

In [ ]:
#model 1- linear regression
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
model=lr.fit(xtrain,ytrain)


#model 2 - k nearest neighbors
from sklearn.neighbors import KNeighborsRegressor
knn=KNeighborsRegressor(n_neighbors=5)
model=knn.fit(xtrain,ytrain)


#model 3 - decision tree regressor
from sklearn.tree import DecisionTreeRegressor
dtr=DecisionTreeRegressor()
model=dtr.fit(xtrain,ytrain)


#model 4 - random forest
from sklearn.ensemble import RandomForestRegressor
rfr=RandomForestRegressor(n_estimators=20)
model=rfr.fit(xtrain,ytrain)


#model 5 - adaboost regressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor
lir=LinearRegression()
abr=AdaBoostRegressor(lir,n_estimators=25)
model=abr.fit(xtrain,ytrain)


#model 6 - support vector machine
from sklearn.svm import SVR
svr=SVR()
model=svr.fit(xtrain,ytrain)


In [ ]:
#model evaluation
from sklearn.metrics import r2_score,mean_absolute_error
models = {
    "Linear Regression": lr,
    "Decision Tree": dtr,
    "Random Forest": rfr,
    "KNN": knn,
    "AdaBoost": abr,
    "SVR": svr
}

for name, model in models.items():
    y_pred = model.predict(xtest)
    print(name)
    print("R2 Score:", r2_score(ytest, y_pred))
    print("MAE:", mean_absolute_error(ytest, y_pred))
    print("-"*30)

#FINAL MODEL - RANDOM FOREST REGRESSOR

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rfr=RandomForestRegressor( n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    random_state=42
)

model=rfr.fit(xtrain,ytrain)

trpred=model.predict(xtrain)
tspred=model.predict(xtest)

from sklearn.metrics import r2_score

trscore=r2_score(ytrain,trpred)
tsscore=r2_score(ytest,tspred)
print('Training Score:',trscore)
print('Testing Score:',tsscore)

In [ ]:
#PICKLE 
with open('model.pkl', 'wb') as file2:
    pickle.dump(model,file2)

In [ ]:
with open('model.pkl' , 'rb') as file4:  
    m= pickle.load(file4)